## We will perform the ingestion process of this pipeline daily on new data from our source (mask PII, track encryption, partition, bucket, and store)
### This involves a manual data load from our source to our Raw Data entry point, but I will build this notebook in a way that checks the current day and utilizes the file naming structure to determine which files to ingest. 
### the file names will use a pre-defined naming convention that contains the partition key (date) for storage.

In [ ]:
import pandas as pd
import json
import datetime
import hashlib

from faker import Faker
from faker_vehicle import VehicleProvider

import os
from os import listdir
from os.path import isfile, join


In [ ]:
current_day = datetime.date.today()
print(current_day)

In [ ]:
#list all file names in the Raw Data folder. 
#ASSUMPTION: They will be prefixed with the correct date for daily batch ingestion, 
# this removes the need to actually load the transaction into dataframs and determine their dates, saving compute time for identifying new files

raw_path = 'Raw Data/'

raw_file_list =  [f for f in listdir(raw_path) if isfile(join(raw_path, f))] #list all file names in the Raw Data folder. They will be prefixed with the date for daily batch ingestion

print(raw_file_list)

In [ ]:
#End goal is to process all files in the Raw Data bucket daily that have not yet been added to the bronze layer. 
#To achieve this, we will need to know the current date, and the last date that the bronze layer was updated.
#Current date is stored above, now identify the last date that the bronze layer was updated.

current_bronze = 'Masked Data_bronze/'

bronze_last_updated = max([datetime.datetime.strptime(f, "%Y-%m-%d").date() for f in listdir(current_bronze) if not isfile(join(current_bronze, f))]) #only search partition folders, not files that may be present

print(bronze_last_updated) #latest date we updated the bronze layer should be 2026-09-30


In [ ]:
#now want to identify files in the raw data folder that were created after the last update to the bronze layer but before the current date
def identify_new_files():
    new_files = [] #store any new files where date was identified via file name prefix
    for filename in raw_file_list:
        try: 
            file_date = datetime.datetime.strptime(filename.split('_txns.csv')[0], '%Y_%m_%d').date()
            if bronze_last_updated < file_date < current_day:
                new_files.append(filename)
        except ValueError: #ignore improperly named daily files
            continue

    return new_files

In [ ]:
#test, identify new files that need to be cleaned/masked for storage in bronze layer
new_files = identify_new_files()
print(new_files)

### what happens if we had more than one day of data that we missed for ingestion into the bronze layer?
#### we can utilize existing batch process from  our full load ipynb, and store all the missed data from new_files into a dataframe for batch processing.
#### the final cell block in raw_to_masked_notebook.ipynb splits the aggregated data into partition and buckets based on the date and location for each txn in the dataframe.
##### Objectives:
1. ~~Using current datetime when runnning this notebook and latest update date of the bronze layer, identify new files in 'Raw Data/' that need to be processed~~
2. Load all new files into dataframe for processing
3. Use cleaning/masking logic from original full load to prepare data for ingestion into storage
4. Store processed new data according to partition/bucket logic in Masked Data_bronze


In [ ]:
# specify folder prefix for concatenating files into dataframe
folder_prefix = 'Raw Data/'

#ensure we navigate to the new files in the correct folder when loading them into the dataframe
full_paths = [os.path.join(folder_prefix, f) for f in new_files if os.path.exists(os.path.join(folder_prefix, f))] 

#load all new files into a single dataframe
#REMINDER: certain fields need explicit cast to string for masking
new_raw_txns = pd.concat([pd.read_csv(f, dtype={'#': 'str', 'Ref No': 'str', 'Reservation ID': 'str', 
                              'User/Preparer': 'str', 'Client Name': 'str', 'Client Fee Override Approved By': 'str',
                                'Client Rate Override Approved By': 'str', 'Approved By': 'str', 'E-mail': 'str', 
                                'Phone': 'str', 'ID Number': 'str', 'Source of Funds': 'str', 
                                'Occupation': 'str', 'Notes': 'str', 'ID Number': 'str'}) 
                                
                                for f in full_paths], 
                                
                                ignore_index=True)

new_raw_txns.head(8)

### We have now completed step 2 of our batch ingestion process: 
##### Objectives:
1. ~~Using current datetime when runnning this notebook and latest update date of the bronze layer, identify new files in 'Raw Data/' that need to be processed~~
2. ~~Load all new files into dataframe for processing~~

#### Remaining tasks:
3. Use cleaning/masking logic from original full load to prepare data for ingestion into storage
4. Store processed new data according to partition/bucket logic in Masked Data_bronze

In [ ]:
# remove blank rows
new_raw_txns = new_raw_txns.dropna(subset=['Ref No'])

In [ ]:
# rename transaction ID column
new_raw_txns = new_raw_txns.rename(columns={'Ref No': 'Transaction ID'})

In [ ]:
#isolate date for each txn for partitioning
new_raw_txns['Date'] = pd.to_datetime(new_raw_txns['Date and Time Entered'], format='mixed').dt.date

In [ ]:
#Remove company name from location
new_raw_txns['Location'] = new_raw_txns['Company - Location'].str[27:]

# remove company for added security and privacy
new_raw_txns = new_raw_txns.drop(['Company - Location'], axis=1)

In [ ]:
#unique identifier for each transaction, needed when a transaction involves multiple currencies.
new_raw_txns['uid'] = new_raw_txns['Transaction ID'].astype(str) + '_' + new_raw_txns['Currency Code'].astype(str)
new_raw_txns = new_raw_txns.set_index(['uid'])

In [ ]:
# NEED TO MODULARIZE THIS APP, add these function to helper file to clean up code

#all the above fields that we want to make or encrypt should be normalized (strip whitespace, convert to lowercase, etc.), 
#so that the masking function can have a consistent input format.

# initiate use of Faker to mask the data deterministically
fake = Faker()

#driver's license ID's will become License Plate numbers
fake.add_provider(VehicleProvider)

# this will be the data we store to the secrets_map file in case of need for decrypting in case of actual fraud detection
location_mapping = {}
name_mapping = {}
company_mapping = {}
email_mapping = {}
phone_mapping = {}
id_number_mapping = {}
fund_source_mapping = {}
occupation_mapping = {}
notes_mapping = {}

#list of mapping for use in case of fraud detection
secrets_map = {'loc': location_mapping, 'name': name_mapping, 'company': company_mapping, 'email': email_mapping, 'phone': phone_mapping, 
               'id_number': id_number_mapping, 'fund_source': fund_source_mapping, 'occupation': occupation_mapping, 'notes': notes_mapping}


#UPDATE SECRETS MAP
# get current mappings -> apply already generated mappings -> add mapping if new input -> overwrite mapping jsons with new mappings
# Load JSON files holding current secrets_map
enc_path = 'encryption/'

for x in secrets_map.keys():
    with open(enc_path + x + '.json', 'r') as f:
        secrets_map[x] = json.load(f) 

#secrets_map now holds all current encryption keys

#helper functions to generate masked data with deterministic seeding while storing mapping of the original values
def get_fake_location(i):
    if i in secrets_map['loc'].keys():
        return secrets_map['loc'][i]
    else:
        fake.seed_instance(i)
        a = fake.city()
        secrets_map['loc'][i] = a
        return a

def get_fake_person_name(i):
    if i in secrets_map['name'].keys():
        return secrets_map['name'][i]
    else:
        fake.seed_instance(i)
        a = fake.name()
        secrets_map['name'][i] = a
        return a

def get_fake_company_name(i):
    if i in secrets_map['company'].keys():
        return secrets_map['company'][i]
    else:
        fake.seed_instance(i)
        a = fake.company()
        secrets_map['company'][i] = a
        return a

def get_fake_email(i):
    if i in secrets_map['email'].keys():
        return secrets_map['email'][i]
    else:
        fake.seed_instance(i)
        a = fake.email()
        secrets_map['email'][i] = a
        return a

def get_fake_phone(i):
    if i in secrets_map['phone'].keys():
        return secrets_map['phone'][i]
    else:
        fake.seed_instance(i)
        a = fake.phone_number()
        secrets_map['phone'][i] = a
        return a

def get_fake_id_number(i):
    if i in secrets_map['id_number'].keys():
        return secrets_map['id_number'][i]
    else:
        fake.seed_instance(i)
        a = fake.license_plate()
        secrets_map['id_number'][i] = a
        return a

def get_fake_fund_source(i):
    if i in secrets_map['fund_source'].keys():
        return secrets_map['fund_source'][i]
    else:
        fake.seed_instance(i)
        a = fake.bs()
        secrets_map['fund_source'][i] = a
        return a

def get_fake_job_title(i):
    if i in secrets_map['occupation'].keys():
        return secrets_map['occupation'][i]
    else:
        fake.seed_instance(i)
        a = fake.job()
        secrets_map['occupation'][i] = a
    return a

def get_fake_notes(i):
    if i in secrets_map['notes'].keys():
        return secrets_map['notes'][i]
    else:
        fake.seed_instance(i)
        a = fake.sentence()
        secrets_map['notes'][i] = a
        return a

#### Findings
- We want to re-use the already generated encryption from the full data load for masking values in the new files that were already masked before
- Need to search the existing encryption jsons for keys that may already exist and use the value for that key when masking the PII of new data
- If the new PII input has not been encrypted/faked before, then add it to the encryption json for future occurences that need masking with the same value

In [ ]:
# iterate over the list of columns to mask and generate our masked dataset
# we will use faker for all columns except ID Number, which will use hashlib
#List of PII columns to iterate over
cols_to_mask = [
    'Location',
    'User/Preparer',
    'Client Name',
    'Client Fee Override Approved By',
    'Client Rate Override Approved By',
    'Company Name',
    'E-mail',
    'Approved By',
    'Phone',
    'ID Number',
    'Source of Funds',
    'Occupation',
    'Notes' 
]


def mask_pii_data(raw_df: pd.DataFrame, PII_cols: list):

    masked_df = raw_df.copy()

    for x in PII_cols:
        if x == 'Location':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_location(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x in ['User/Preparer', 'Client Name', 'Client Fee Override Approved By', 'Client Rate Override Approved By', 'Approved By']:
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_person_name(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'Company Name':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_company_name(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'E-mail':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_email(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'Phone':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_phone(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'ID Number':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            # use hashlib for ID Number
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: hashlib.sha256(y.encode('utf-8')).hexdigest() if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'Source of Funds':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_fund_source(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'Occupation':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_job_title(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'Notes':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_notes(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

    return masked_df    

In [ ]:
new_masked_txns = mask_pii_data(new_raw_txns, cols_to_mask)

In [ ]:
# NEED TO MODULARIZE THIS APP, add these function to helper file to clean up code
# overwrite secrets map with any new masked values encountered in batch
def write_secrets_map(folder: str):
    for i in secrets_map.keys():
        with open(folder + "/" + str(i) + ".json", "w") as f:
            json.dump(secrets_map[i], f, indent=4, sort_keys=True)

write_secrets_map('encryption')

### We have implemented the reusing of previous mappings by explicitly referencing our encryption dictionaries from previous runs of the data ingestion / PII masking

#### we have now completed task 3 of our new data ingestion process
3. ~~Use cleaning/masking logic from original full load to prepare data for ingestion into storage~~

#### Remaining tasks:
4. Store processed new data according to partition/bucket logic in Masked Data_bronze

In [ ]:
#set destination folder for the partitioned and bucketed data
bronze_layer = "Masked Data_bronze/"

new_masked_txns.groupby(['Date', 'Location']).first()

In [ ]:
for (date, loc), group in new_masked_txns.groupby(['Date', 'Location']):
    if not os.path.exists(bronze_layer + str(date) + '/' + str(loc)):
        os.makedirs(bronze_layer + str(date) + '/' + str(loc))
        group.to_csv(bronze_layer + str(date) + '/' + str(loc) + '/' + str(date) + '_' + str(loc) + '_' + 'data.csv', index=True) #index must be included for uniqueness